In [ ]:
#| hide
from bqdf.core import *

# bqdf

> A thin, ergonomic wrapper around pandas_gbq that handles type conversions automatically and provides cleaner syntax for BigQuery operations

This file will become your README and also the index of your documentation.

## Developer Guide

If you are new to using `nbdev` here are some useful pointers to get you started.

### Install bqdf in Development mode

```sh
# make sure bqdf package is installed in development mode
$ pip install -e .

# make changes under nbs/ directory
# ...

# compile to have changes apply to bqdf
$ nbdev_prepare
```

## Usage

### Installation

Install latest from the GitHub [repository][repo]:

```sh
$ pip install git+https://github.com/motdam/bqdf.git
```

or from [conda][conda]

```sh
$ conda install -c motdam bqdf
```

or from [pypi][pypi]


```sh
$ pip install bqdf
```


[repo]: https://github.com/motdam/bqdf
[docs]: https://motdam.github.io/bqdf/
[pypi]: https://pypi.org/project/bqdf/
[conda]: https://anaconda.org/motdam/bqdf

### Documentation

Documentation can be found hosted on this GitHub [repository][repo]'s [pages][docs]. Additionally you can find package manager specific guidelines on [conda][conda] and [pypi][pypi] respectively.

[repo]: https://github.com/motdam/bqdf
[docs]: https://motdam.github.io/bqdf/
[pypi]: https://pypi.org/project/bqdf/
[conda]: https://anaconda.org/motdam/bqdf

## How to use

This lib provides convenience functions for streamlining the interface of the pandas-gbq library to perform CRUD operations in BigQuery more quickly

In [ ]:
from google.oauth2 import service_account
import os
import json

In [ ]:

#| hide
import os
if 'CI' not in os.environ:  # CI is set automatically in GitHub Actions
    credentials = service_account.Credentials.from_service_account_info(
        json.loads(os.environ['GOOGLE_APPLICATION_CREDENTIALS'])
    )


In [ ]:

gn_query = """
-- todays top 10 search terms in England
SELECT refresh_date, rank, term, score, percent_gain / 100 as percent_gain, country_name, week
FROM `bigquery-public-data.google_trends.international_top_rising_terms` 
WHERE country_name = 'United Kingdom'
  and refresh_date = current_date - 1
  and region_name = 'England'
order by refresh_date desc, week desc, rank
limit 10
"""

### Reading a bq table

In [ ]:
#| eval: false
df = read(gn_query, credentials=credentials, project_id='bq-sandbox-motdam')
df

What do you think fo my example, do you think it is wokring okay or there woul dbe better things to add?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Your example looks good! It demonstrates the core functionality clearly with a relatable query (top girls names). The output shows it's working and includes helpful progress feedback.

A few thoughts:

**What works well:**
- The query is simple and the results are easy to understand
- The output shows timing and data info, which is useful
- The comment helps explain what the query does

**Possible additions:**
- You could show an example without credentials for users with default auth set up
- Maybe demonstrate error handling or what happens with a bad query?
- A comparison showing how much simpler this is than using pandas_gbq directly?

What aspect are you most interested in improving? The documentation clarity, the example variety, or something about the functionality itself?

To recreate the above with the original library you would need the below boiler plate to inspect the results and convert the type fields properly.

In [ ]:
#| eval: false
import pandas_gbq
import pandas as pd

df = pandas_gbq.read_gbq(gn_query, project_id='bq-sandbox-motdam', credentials=credentials)
df = df.astype({
    'percent_gain':'Float64'
})
for d in ['week', 'refresh_date']:
    df[d] = pd.to_datetime(df[d])
print(df.info())
df

How is that looking?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Perfect! This comparison really highlights the value of your library. The contrast is clear:

**With `bqdf.read()`:**
- One function call
- Automatic type handling (including the `dbdate` type)
- Built-in timing and `.info()` display

**With `pandas_gbq` directly:**
- Multiple manual type conversions
- No automatic inspection
- More verbose boilerplate

This side-by-side example would be excellent in your README. It immediately shows developers why they'd want to use your library.

One small note: I see your library shows "Retrieved 10 rows × 7 cols in 0.95s" which is nice feedback that the raw pandas_gbq doesn't provide.

Do you want to add this comparison to your documentation, or are you thinking about what else to showcase?

The rest `to` function is unchanged beyond removing the redundant _gbq suffix. We can write our df back into BigQuery using hte `to` function.

In [ ]:
#| eval: false
# Write the dataframe to a temporary table
to(df, 'bq-sandbox-motdam.temporary.top_10_eng_search_terms', credentials=credentials, if_exists='replace')


Please generate an example of using execute, how could we use that here?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's an example using `execute` that builds on your current workflow. You could use it to create or modify the table structure before writing:

```python
# Create the table with specific schema if it doesn't exist
execute("""
CREATE TABLE IF NOT EXISTS `bq-sandbox-motdam.temporary.top_10_eng_search_terms` (
  week DATE,
  rank INT64,
  term STRING,
  score INT64,
  percent_gain FLOAT64,
  country_name STRING,
  refresh_date TIMESTAMP
)
""", credentials=credentials, project_id='bq-sandbox-motdam')
```

Or you could use it to clean up old data:

```python
# Delete old entries before inserting new ones
execute("""
DELETE FROM `bq-sandbox-motdam.temporary.top_10_eng_search_terms`
WHERE refresh_date < CURRENT_DATE() - 30
""", credentials=credentials, project_id='bq-sandbox-motdam')
```

Would either of these examples work for your documentation, or did you have a different use case in mind?

is there another, simpoer one?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Sure, here's a simpler example:

```python
execute("DROP TABLE IF EXISTS `bq-sandbox-motdam.temporary.top_10_eng_search_terms`", 
        credentials=credentials, project_id='bq-sandbox-motdam')
```

This just drops the temporary table you created. Would you like me to show an even simpler one?

The `ex` enables non df based CRUD operations within the same api which can be useful for creating feature processing pipelines.


In [ ]:
#| eval: false
project = 'bq-sandbox-motdam'

def create_top_terms(period, days):
    return f"""
    CREATE OR REPLACE TABLE `{project}.temporary.top_terms_{period}` AS
    WITH ranked AS (
      SELECT region_name, term, COUNT(*) as appearances, AVG(rank) as avg_rank,
        ROW_NUMBER() OVER (PARTITION BY region_name ORDER BY COUNT(*) DESC, AVG(rank)) as rn
      FROM `bigquery-public-data.google_trends.international_top_rising_terms`
      WHERE country_name = 'United Kingdom'
        AND region_name IN ('England', 'Scotland', 'Wales', 'Northern Ireland')
        AND refresh_date BETWEEN CURRENT_DATE() - {days} AND CURRENT_DATE()
        AND rank <= 100
      GROUP BY region_name, term
    )
    SELECT region_name, term as top_term_{period}
    FROM ranked WHERE rn = 1
    """

ex(create_top_terms('today', 1), credentials=credentials, project_id=project)
ex(create_top_terms('week', 8), credentials=credentials, project_id=project)
ex(create_top_terms('month', 31), credentials=credentials, project_id=project)
ex(create_top_terms('year', 366), credentials=credentials, project_id=project)

final_query = f"""
SELECT t.region_name, t.top_term_today, w.top_term_week, m.top_term_month, y.top_term_year
FROM `{project}.temporary.top_terms_today` as t
JOIN `{project}.temporary.top_terms_week` as w ON t.region_name = w.region_name
JOIN `{project}.temporary.top_terms_month` as m ON t.region_name = m.region_name
JOIN `{project}.temporary.top_terms_year` as y ON t.region_name = y.region_name
ORDER BY t.region_name
"""

read(final_query, credentials=credentials, project_id=project)

British search history in a nutshell: 'Is it raining?' followed immediately by 'Can I afford to move somewhere sunny?